# 03d - Análisis final con BERTopic etiquetado

Esta notebook carga el análisis de tópicos que usamos en la entrega final del práctico 3. No vuelve a entrenar el modelo: parte de `df_final_with_topics.csv` y `topic_document_info.csv`, aplica los labels definidos manualmente y reconstruye las visualizaciones principales para presentación/portfolio.

## Carga de resultados finales

El entrenamiento completo y las pruebas exploratorias quedan en las notebooks anteriores. Acá trabajamos con las salidas ya consolidadas: tópico asignado, probabilidad, palabras representativas y texto del documento para inspección cualitativa.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

try:
    from umap import UMAP
except Exception:
    UMAP = None

PROCESSED_DIR = Path("../data/processed") if Path.cwd().name == "notebooks" else Path("data/processed")
REPORTS_DIR = Path("../reports") if Path.cwd().name == "notebooks" else Path("reports")

DF_PATH = PROCESSED_DIR / "df_final_with_topics.csv"
TOPIC_INFO_PATH = PROCESSED_DIR / "topic_document_info.csv"
COORDS_PATH = PROCESSED_DIR / "portfolio_umap_2d.csv"

custom_labels = {
    0: "Críticas y burlas sobre el sobrepeso",
    1: "La obesidad como enfermedad",
    2: "Comida y sobrepeso",
    3: "Números, peso y kilos",
    4: "Negaciones",
    5: "Nacionalidad y estereotipos culturales",
    6: "Autopercepción",
    7: "La obesidad como insulto y descalificación política",
    8: "Gatos y obesidad",
    9: "Salud, sobrepeso y ejercicio físico",
    10: "Obesidad y diabetes",
    11: "Alimentación, salud, políticas de estado",
    12: "La obesidad y sus complicaciones clínicas",
    13: "Debates de género sobre apariencia y sobrepeso",
    14: "La obesidad como pandemia global",
    15: "Insultos directos y comparaciones",
    16: "Ella y su obesidad",
    17: "Descalificaciones personales y agresiones directas",
    18: "Confesiones sobre comida y apetito",
    19: "El sobrepeso en el mundo del deporte",
}

selected_topics = list(custom_labels)


In [2]:
df = pd.read_csv(DF_PATH)
topic_document_info = pd.read_csv(TOPIC_INFO_PATH)

df["topic"] = pd.to_numeric(df["topic"], errors="coerce").fillna(-1).astype(int)
df["topic_prob"] = pd.to_numeric(df["topic_prob"], errors="coerce")
df["topic_label"] = df["topic"].map(custom_labels).fillna("Otros / outliers")

df.shape, topic_document_info.shape

((8882, 69), (8882, 10))

## Resumen del modelo final

Nos quedamos con BERTopic porque produjo grupos más interpretables que KMeans sobre TF-IDF/Word2Vec/Sentence-BERT y que los modelos clásicos NMF/LDA. La decisión no se apoya en una métrica única: combina revisión cualitativa, palabras representativas, ejemplos y métricas de evaluación agregadas.

In [3]:
summary = pd.DataFrame({
    "documentos": [len(df)],
    "tópicos seleccionados": [len(selected_topics)],
    "outliers / otros": [int((df["topic"] == -1).sum())],
    "probabilidad media": [df.loc[df["topic"].isin(selected_topics), "topic_prob"].mean()],
    "probabilidad mediana": [df.loc[df["topic"].isin(selected_topics), "topic_prob"].median()],
})
summary

,documentos,tópicos seleccionados,outliers / otros,probabilidad media,probabilidad mediana
0,8882,20,3348,0.57802,0.426151


In [4]:
metrics_path = REPORTS_DIR / "topic_evaluation_metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)
else:
    print("No se encontró topic_evaluation_metrics.csv")

,model,n_topics,n_outliers,outlier_rate,topic_size_min,topic_size_median,topic_size_max,silhouette_cosine,davies_bouldin,calinski_harabasz,topic_diversity_top10,mean_npmi_top10,topic_probability_mean,topic_probability_median
0,BERTopic existente,28,3219,0.374389,67,146.0,461,-0.041341,9.195329,14.802547,0.525000,0.117346,0.641842,1.0
1,BERTopic ajustado prob>=0.35,28,5351,0.622354,67,118.0,137,-0.026924,8.180403,13.624878,0.378571,-0.001025,0.950797,1.0
2,KMeans TF-IDF+SVD k=28,28,0,0.000000,127,225.5,955,0.215785,2.678227,175.859338,0.507143,0.005539,NaN,NaN
3,KMeans TF-IDF+SVD compacto k=12,12,0,0.000000,175,592.5,1672,0.180136,3.012669,250.130392,0.558333,0.070049,NaN,NaN


## Tópicos y palabras representativas

Los labels son interpretativos: se definieron mirando palabras, documentos representativos y ejemplos por tópico. La tabla permite auditar rápidamente qué términos empujan cada lectura.

In [5]:
topic_info_unique = (
    topic_document_info[topic_document_info["Topic"].isin(selected_topics)]
    .drop_duplicates("Topic")
    .copy()
)
topic_info_unique["label"] = topic_info_unique["Topic"].map(custom_labels)
topic_counts = df[df["topic"].isin(selected_topics)]["topic"].value_counts().rename("count")

topic_table = (
    topic_info_unique[["Topic", "label", "Top_n_words", "Representation", "KeyBERT", "MMR"]]
    .merge(topic_counts, left_on="Topic", right_index=True, how="left")
    .sort_values("Topic")
)
topic_table[["Topic", "label", "count", "Top_n_words"]]

,Topic,label,count,Top_n_words
21,0,Críticas y burlas sobre el sobrepeso,477,hizo - hablar - puta - gran - hombre - final -...
27,1,La obesidad como enfermedad,461,enfermedad - salud - cuerpo - mental - enferme...
4,2,Comida y sobrepeso,340,comida - comer - come - mala - podes - mucha -...
0,3,"Números, peso y kilos",332,kilos - bajar - 20 - 10 - 50 - normal - 30 - i...
88,4,Negaciones,298,quiere - medio - podes - tampoco - viendo - bo...
9,5,Nacionalidad y estereotipos culturales,291,argentina - país - final - mayoría - comida - ...
6,6,Autopercepción,276,amigo - imc - 10 - veo - puedo - xq - momento ...
10,7,La obesidad como insulto y descalificación pol...,263,milei - ustedes - hablando - tiempo - poder - ...
64,8,Gatos y obesidad,215,pobre - bajo - puedo - meses - comida - pq - c...
53,9,"Salud, sobrepeso y ejercicio físico",180,ir - buen - pueden - cuerpo - bajar - forma - ...


## Mapa 2D de documentos

La reducción se usa para explorar la estructura del corpus, no como prueba definitiva de separación. En este corpus los documentos aparecen bastante mezclados; eso es parte del resultado y ayuda a explicar por qué la interpretación cualitativa sigue siendo necesaria.

In [6]:
STOPWORDS = [
    "a", "al", "algo", "ante", "antes", "como", "con", "contra", "cual", "cuando",
    "de", "del", "desde", "donde", "durante", "e", "el", "ella", "ellas", "ellos",
    "en", "entre", "era", "eran", "es", "esa", "esas", "ese", "eso", "esos",
    "esta", "estan", "estar", "este", "esto", "estos", "fue", "fueron", "ha",
    "han", "hasta", "hay", "la", "las", "le", "les", "lo", "los", "mas", "me",
    "mi", "mis", "muy", "no", "nos", "o", "para", "pero", "por", "porque",
    "que", "se", "si", "sin", "son", "su", "sus", "te", "tiene", "todo",
    "un", "una", "unas", "uno", "unos", "vos", "y", "ya", "rt", "link_url",
]

def build_coords(texts):
    vectorizer = TfidfVectorizer(
        max_features=7000,
        min_df=3,
        max_df=0.85,
        ngram_range=(1, 2),
        stop_words=STOPWORDS,
        strip_accents="unicode",
    )
    x_tfidf = vectorizer.fit_transform(texts.fillna(""))
    reduced = normalize(TruncatedSVD(n_components=min(80, x_tfidf.shape[1] - 1), random_state=42).fit_transform(x_tfidf))
    if UMAP is not None:
        coords_array = UMAP(n_neighbors=12, n_components=2, min_dist=0.05, metric="cosine", random_state=42, low_memory=False).fit_transform(reduced)
    else:
        coords_array = TruncatedSVD(n_components=2, random_state=42).fit_transform(reduced)
    return pd.DataFrame({"doc_id": np.arange(len(texts)), "x": coords_array[:, 0], "y": coords_array[:, 1]})

if COORDS_PATH.exists():
    coords = pd.read_csv(COORDS_PATH)
else:
    coords = build_coords(df["rawContent_clean"])
    coords.to_csv(COORDS_PATH, index=False)

plot_df = df.reset_index(names="doc_id").merge(coords, on="doc_id", how="left")
plot_df["topic_label_short"] = plot_df["topic"].map(lambda t: f"{t}: {custom_labels[t]}" if t in custom_labels else "Otros / outliers")
plot_df["hover_text"] = plot_df["document"].fillna(plot_df["rawContent_clean"]).astype(str).str.slice(0, 220)

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="topic_label_short",
    hover_data={"hover_text": True, "topic_prob": ":.3f", "x": False, "y": False},
    opacity=0.72,
    height=680,
    title="Mapa 2D de documentos coloreado por tópico final",
)
fig.update_traces(marker={"size": 5})
fig.update_layout(legend_title_text="Tópico", margin={"l": 20, "r": 20, "t": 60, "b": 20})
fig.show()

## Tamaño de tópicos y sentimiento

Estos gráficos resumen la distribución del modelo elegido y permiten detectar tópicos grandes, pequeños o especialmente cargados de sentimiento negativo/hostilidad.

In [7]:
topic_sizes = (
    df[df["topic"].isin(selected_topics)]
    .groupby(["topic", "topic_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
fig = px.bar(topic_sizes, x="count", y="topic_label", orientation="h", title="Cantidad de documentos por tópico final")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=650, margin={"l": 20, "r": 20, "t": 60, "b": 20})
fig.show()

In [8]:
sentiment_by_topic = (
    df[df["topic"].isin(selected_topics)]
    .groupby(["topic", "topic_label", "pysentimiento"])
    .size()
    .reset_index(name="count")
)
fig = px.bar(
    sentiment_by_topic,
    x="topic_label",
    y="count",
    color="pysentimiento",
    title="Sentimiento por tópico",
    barmode="stack",
)
fig.update_layout(xaxis_tickangle=-35, height=620, margin={"l": 20, "r": 20, "t": 60, "b": 160})
fig.show()

## Lectura cualitativa de tópicos

La interpretación final salió de mirar palabras, documentos de alta probabilidad y ejemplos concretos. La función siguiente permite revisar un tópico sin volver a tocar el modelo.

In [9]:
topic_descriptions = {
    0: "Comentarios que usan el sobrepeso como insulto o forma de descalificación hacia figuras del deporte, la política o el entretenimiento.",
    1: "Debates sobre la obesidad como condición médica y sus riesgos, con críticas a la romantización del sobrepeso.",
    2: "Comentarios que vinculan alimentos, formas de comer y sobrepeso, muchas veces con tono de burla o estereotipo.",
    3: "Testimonios y discusiones con cifras exactas: kilos, altura, IMC y comparaciones con tablas de peso ideal.",
    4: "Agrupa negaciones y respuestas defensivas. Es un tópico menos sustantivo, útil como límite interpretativo del modelo.",
    5: "Conversaciones que relacionan obesidad, países, nacionalidad y estereotipos culturales.",
    6: "Mensajes en primera persona sobre autopercepción corporal, humor, frustración o resignación.",
    7: "Uso del sobrepeso como herramienta de ataque en conversaciones políticas.",
    8: "Comentarios sobre gatos domésticos con sobrepeso, generalmente afectuosos o humorísticos.",
    9: "Debates sobre sobrepeso, actividad física, entrenamiento, articulaciones y salud.",
    10: "Relación entre obesidad y diabetes, tanto en registros médicos como en comentarios agresivos.",
    11: "Conversaciones sobre alimentación, salud pública, políticas de estado y condiciones sociales.",
    12: "Mirada clínica sobre comorbilidades y enfermedades crónicas asociadas a la obesidad.",
    13: "Debates de género sobre apariencia física, críticas al cuerpo y acusaciones de doble estándar.",
    14: "Tratamiento de la obesidad como problema global, epidemia o pandemia con foco informativo.",
    15: "Insultos directos y comparaciones donde el sobrepeso funciona como descalificativo central.",
    16: "Comentarios dirigidos a terceras personas, especialmente mujeres mediáticas o de reality shows.",
    17: "Agresiones directas en segunda persona, donde el cuerpo se usa para atacar carácter, moral o inteligencia.",
    18: "Confesiones personales sobre comida, apetito, placer y culpa asociada al sobrepeso.",
    19: "Críticas deportivas, sobre todo fútbol, donde el peso se asocia a bajo rendimiento o falta de profesionalismo.",
}

def inspect_topic(topic_id, n_documents=8):
    if topic_id not in custom_labels:
        raise ValueError("El tópico no está dentro del conjunto interpretado 0-19")
    display(Markdown(f"### Tópico {topic_id}: {custom_labels[topic_id]}"))
    display(Markdown(topic_descriptions[topic_id]))
    display(topic_table.loc[topic_table["Topic"] == topic_id, ["Top_n_words", "KeyBERT", "MMR"]])
    cols = ["topic_prob", "pysentimiento", "hs_spanlp", "document"]
    examples = (
        df[df["topic"] == topic_id]
        .sort_values("topic_prob", ascending=False)
        [cols]
        .head(n_documents)
    )
    display(examples)

inspect_topic(13, n_documents=8)

### Tópico 13: Debates de género sobre apariencia y sobrepeso

Debates de género sobre apariencia física, críticas al cuerpo y acusaciones de doble estándar.

,Top_n_words,KeyBERT,MMR
134,mujer - cuerpo - chica - diciendo - hombre - u...,"['mujer', 'puta', 'diciendo', 'hablando', 'san...","['ustedes', 'amiga', 'mina', 'pedazo', 'video'..."


,topic_prob,pysentimiento,hs_spanlp,document
134,1.0,NEG,True,Mujeres cuando una mujer hace chistes sobre el...
135,1.0,NEG,True,Mujeres cuando una mujer haciendo chistes sobr...
280,1.0,NEG,True,Hace cuántas décadas que no ves un pene ? Hace...
320,1.0,NEG,False,"Entonces te teñís de colorada, es lo mismo. Yo..."
343,1.0,NEG,True,"Estéticamente mal, ser sexy y ser obesa no van..."
374,1.0,NEG,False,le decís quasimodo y espanto a un hombre obeso...
382,1.0,NEG,False,"Donde esta el sobrepeso? Pasa foto de cuerpo ""..."
404,1.0,NEG,False,tiene infinidad de complejos con su apariencia...


## Export para la página

La presentación web de GitHub Pages consume `data/public/portfolio_payload.json`. Ese archivo se genera con `scripts/build_portfolio_payload.py` y permite mostrar este análisis final junto con versiones comparativas de KMeans, NMF, LDA y BERTopic.

In [10]:
payload_path = Path("../data/public/portfolio_payload.json") if Path.cwd().name == "notebooks" else Path("data/public/portfolio_payload.json")
print(payload_path)
print("Existe:", payload_path.exists())

../data/public/portfolio_payload.json
Existe: True
